In [ ]:
import requests
from google.transit import gtfs_realtime_pb2
import time
import pandas as pd
from datetime import datetime
import os

# --- CONFIGURATION ---
API_KEY = "h7i00LqgTiaaReuCvKy6TndN4xJrMR1y"  # Ensure this matches your OTD Key
DATA_FOLDER = "delhi_bus_data"                 
INTERVAL = 60                                  

# 1. Create the folder if it doesn't exist
if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)
    print(f"📂 Data folder ready: /{DATA_FOLDER}/")

# The URL for live data
URL = f"https://otd.delhi.gov.in/api/realtime/VehiclePositions.pb?key={API_KEY}"

print(f"🔴 STARTED TRACKING. Data collection active.")
print(f"ℹ️  Files will split automatically every hour.")
print("Press 'Ctrl + C' in the terminal to stop safely.\n")

try:
    while True:
        # 2. Capture Time
        now = datetime.now()
        current_timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
        
        # 3. GENERATE FILENAME BASED ON THE CURRENT HOUR
        # Example: bus_data_2025-12-18_0800.csv
        file_name = f"bus_data_{now.strftime('%Y-%m-%d_%H00')}.csv"
        file_path = os.path.join(DATA_FOLDER, file_name)

        print(f"[{current_timestamp}] Fetching data -> {file_name}...", end=" ")

        try:
            response = requests.get(URL, timeout=10)
            
            if response.status_code == 200:
                feed = gtfs_realtime_pb2.FeedMessage()
                feed.ParseFromString(response.content)
                
                bus_list = []
                for entity in feed.entity:
                    if entity.HasField('vehicle'):
                        # This is the "Row" for one bus
                        bus_data = {
                            "timestamp": current_timestamp,
                            "bus_id": entity.id,
                            "latitude": entity.vehicle.position.latitude,
                            "longitude": entity.vehicle.position.longitude,
                            "speed": entity.vehicle.position.speed,
                            "trip_id": entity.trip_update.trip.trip_id if entity.HasField('trip_update') else "N/A",
                            "route_id": entity.trip_update.trip.route_id if entity.HasField('trip_update') else "N/A"
                        }
                        bus_list.append(bus_data)
                
                # 4. Save to CSV
                if bus_list:
                    df = pd.DataFrame(bus_list)
                    
                    # If file doesn't exist for this hour, create it with headers.
                    # If it DOES exist, append without headers.
                    if not os.path.exists(file_path):
                        df.to_csv(file_path, index=False, mode='w')
                    else:
                        df.to_csv(file_path, index=False, mode='a', header=False)
                    
                    print(f"✅ Saved {len(bus_list)} rows.")
                else:
                    print("⚠️ No buses found (Empty Feed).")
            
            else:
                print(f"❌ Server Error: {response.status_code}")

        except Exception as e:
            print(f"❌ Error: {e}")

        # 5. Wait for 60 seconds
        time.sleep(INTERVAL)

except KeyboardInterrupt:
    print("\n🛑 TRACKING STOPPED.")

🔴 STARTED TRACKING. Data collection active.
ℹ️  Files will split automatically every hour.
Press 'Ctrl + C' in the terminal to stop safely.

[2025-12-22 06:57:41] Fetching data -> bus_data_2025-12-22_0600.csv... ✅ Saved 1935 rows.
[2025-12-22 06:58:42] Fetching data -> bus_data_2025-12-22_0600.csv... ✅ Saved 1950 rows.
[2025-12-22 06:59:42] Fetching data -> bus_data_2025-12-22_0600.csv... ✅ Saved 1983 rows.
[2025-12-22 07:00:43] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 2000 rows.
[2025-12-22 07:01:44] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 1995 rows.
[2025-12-22 07:02:45] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 2000 rows.
[2025-12-22 07:03:45] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 2021 rows.
[2025-12-22 07:04:46] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 2041 rows.
[2025-12-22 07:05:47] Fetching data -> bus_data_2025-12-22_0700.csv... ✅ Saved 2042 rows.
[2025-12-22 07:06:48] Fetching data -> bus_data_2